# 08 — Alerts, incidents and dying-gasp context

This notebook converts the frozen development scores into alert intervals and
consolidated incidents. It does not read fault labels. A dying-gasp signal, if
the operator supplies one as an operational event, is used only as incident
context; it is never fabricated from continuous telemetry and never changes
the frozen anomaly threshold.

By default this notebook stops if Notebook 07 did not select a configuration.
Set `ALLOW_DIAGNOSTIC_DEMO=1` only for a clearly labelled internal demo.


## 1. Setup and frozen configuration


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.detectors import alerts_from_score_file
from telco_anomaly.evaluation import form_cases
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
MODEL_RUN_ID = os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v5")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v5")
INCIDENT_RUN_ID = os.getenv("PON_INCIDENT_RUN_ID", "synthetic_pon_incidents_v5")

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
SELECTION_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "incidents" / "synthetic_pon" / INCIDENT_RUN_ID

allow_diagnostic = os.getenv("ALLOW_DIAGNOSTIC_DEMO", "0") == "1"
selected_path = SELECTION_ROOT / "selected_configuration.json"
if selected_path.exists():
    configuration_path = selected_path
elif allow_diagnostic:
    configuration_path = SELECTION_ROOT / "best_diagnostic_configuration.json"
else:
    raise RuntimeError(
        "Notebook 07 wrote no selected configuration. Keep holdout sealed and improve "
        "the detector, or set ALLOW_DIAGNOSTIC_DEMO=1 for a non-deployable demo."
    )

configuration = read_json(configuration_path)
core_manifest = read_json(CORE_ROOT / "manifest.json")
model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
require_same(model_manifest, core_fingerprint=core_manifest["fingerprint"])
if file_sha256(PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py") != model_manifest["detectors_module_sha256"]:
    raise ValueError("Detector code changed after the model was fitted")
require_same(
    configuration,
    model_manifest_sha256=file_sha256(MODEL_ROOT / "model_manifest.json"),
    resolved_policy_sha256=model_manifest["resolved_policy_sha256"],
    evaluation_module_sha256=file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
    ),
)
score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()

display(pd.Series({
    "configuration_status": configuration["status"],
    "portfolio": configuration["candidate"],
    "channels": configuration["channels"],
    "truth_opened": False,
}, name="value").to_frame())


## 2. Convert scores to persistent alerts


In [ ]:
alert_frames = []
for channel in configuration["channels"]:
    alert_frames.append(alerts_from_score_file(
        score_path,
        channel,
        configuration["thresholds"][channel],
        min_consecutive=configuration["persistence_observations"][channel],
        recovery_consecutive=configuration["recovery_observations"],
    ))

alerts = pd.concat(alert_frames, ignore_index=True).sort_values("alert_start")
alerts = alerts.reset_index(drop=True)
alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]

display(pd.Series({
    "alerts": len(alerts),
    "affected_entities": alerts["entity_id"].nunique() if len(alerts) else 0,
    "channels": alerts["model_id"].nunique() if len(alerts) else 0,
}, name="count").to_frame())
display(alerts.head(20))


## 3. Consolidate alerts into operator-facing incidents


In [ ]:
cases, members = form_cases(
    alerts,
    topology,
    gap_seconds=configuration["incident_quiet_period_seconds"],
    thresholds=configuration["thresholds"],
    shared_scope_models=("group_common_mode",),
)
cases = cases.sort_values(
    ["anomaly_evidence_score", "case_start"], ascending=[False, True]
).reset_index(drop=True)
cases.insert(0, "rank", np.arange(1, len(cases) + 1))
display(cases.head(30))


## 4. Attach optional dying-gasp evidence without changing detection


In [ ]:
events_path = CORE_ROOT / "operational_events.parquet"
dying_gasp = pd.DataFrame()
if events_path.exists():
    operational_events = pd.read_parquet(events_path)
    dying_gasp = operational_events.loc[
        operational_events["event_family"].isin(["dying_gasp", "power_loss"])
    ].copy()

cases["dying_gasp_events"] = 0
if len(dying_gasp) and len(cases):
    event_members = members.merge(
        alerts[["alert_id", "entity_id"]], on=["alert_id", "entity_id"], how="left"
    )[["case_id", "entity_id"]].drop_duplicates()
    evidence = event_members.merge(dying_gasp, on="entity_id", how="inner")
    evidence = evidence.merge(cases[["case_id", "case_start", "case_end"]], on="case_id")
    within = evidence["event_ts"].between(
        evidence["case_start"] - pd.Timedelta(hours=1),
        evidence["case_end"] + pd.Timedelta(hours=1),
    )
    counts = evidence.loc[within].groupby("case_id").size()
    cases["dying_gasp_events"] = cases["case_id"].map(counts).fillna(0).astype(int)
    dying_gasp_status = "attached_to_alerted_entities_as_context_only"
else:
    dying_gasp_status = "not_available_not_inferred"

print("Dying-gasp status:", dying_gasp_status)


## 5. Save the minimal operational result


In [ ]:
incident_manifest = {
    "dataset": "synthetic_pon",
    "partition": "development",
    "configuration_status": configuration["status"],
    "configuration_file": configuration_path.name,
    "configuration_sha256": file_sha256(configuration_path),
    "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
    "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
    "core_fingerprint": core_manifest["fingerprint"],
    "alerts": len(alerts),
    "incidents": len(cases),
    "dying_gasp_status": dying_gasp_status,
    "truth_files_read": [],
}

if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "incident_manifest.json")
    require_same(
        previous,
        configuration_sha256=incident_manifest["configuration_sha256"],
        model_manifest_sha256=incident_manifest["model_manifest_sha256"],
        resolved_policy_sha256=incident_manifest["resolved_policy_sha256"],
        core_fingerprint=incident_manifest["core_fingerprint"],
    )
    print("Using existing immutable incidents:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        alerts.to_parquet(output / "alerts.parquet", index=False)
        cases.to_parquet(output / "incidents.parquet", index=False)
        members.to_parquet(output / "incident_members.parquet", index=False)
        write_json(output / "incident_manifest.json", incident_manifest)

assert incident_manifest["truth_files_read"] == []
print("PASS — alert consolidation did not use truth")
print("Next: 09_TOPOLOGY_LOCALISATION.ipynb")
